# 03 — Generation Agent: Dense GRPO from SFT

Inputs: the **trained SFT adapter checkpoint** from Notebook 02, training questions/schema/gold SQL, reasoning bank/evidence and SQLite databases. Model responses are sampled independently of gold labels; gold is available only to reward calculation. Output: optimized adapter with optimizer/RNG state. Use 3B first; 7B is an optional memory-sensitive experiment, not a promised T4 configuration.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR, 'grpo_3b')  # Cold-kernel bootstrap; smoke stays offline.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'grpo_3b'
import json
OUTPUT_DIR = paths.output(PHASE)
from src.training.kaggle import initialize_environment
environment_checks = initialize_environment(REPO_DIR, PHASE, OUTPUT_DIR, smoke=SMOKE, device=paths.device)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or os.environ.get("CHECKPOINT_RESUME_SOURCE", "none")
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})

from src.training.kaggle import initialize_checkpoint_backend
checkpoint_backend = initialize_checkpoint_backend(OUTPUT_DIR, PHASE, smoke=SMOKE)


Set `SFT_CHECKPOINT` to an attached Notebook 02 checkpoint directory, `GRPO_DATA` to training JSONL and `DB_DIR` to databases. Begin with `GROUP_SIZE=2`, `MAX_NEW_TOKENS=128`. Constant-reward groups carry zero advantages and provide no policy gradient; logs expose this rather than inventing reward variance.

In [ ]:
from src.training.train_grpo_generator import run_grpo_training
GRPO_DATA = paths.input("GRPO_DATA", "mini_bird.jsonl" if SMOKE else "grpo_train.jsonl")
DB_DIR = paths.input("DB_DIR", "." if SMOKE else "databases")
SFT_CHECKPOINT = os.environ.get("SFT_CHECKPOINT")
summary = run_grpo_training(data_path=GRPO_DATA, db_dir=DB_DIR,
    output_dir=OUTPUT_DIR, dry_run=SMOKE, sft_checkpoint=SFT_CHECKPOINT,
    device=paths.device, max_steps=MAX_STEPS, max_length=MAX_LENGTH, save_steps=SAVE_STEPS,
    seed=SEED, num_rollouts=int(os.environ.get("GROUP_SIZE", "2")),
    max_new_tokens=int(os.environ.get("MAX_NEW_TOKENS", "128")),
    gradient_accumulation_steps=int(os.environ.get("GRAD_ACCUM", "8")),
    resume_from_checkpoint=RESUME)
print(json.dumps(summary, indent=2))


## Đồng bộ checkpoint và khôi phục

Trước setup, cấu hình repo HF **private**, Secret `HF_TOKEN`, `CHECKPOINT_SYNC_ENABLED=1`,
`CHECKPOINT_HF_REPO_ID`, `CHECKPOINT_EXPERIMENT_ID` riêng cho mỗi thí nghiệm và
`CHECKPOINT_RESUME_SOURCE=auto`. Xem hướng dẫn tiếng Việt trong `KAGGLE.md`.
Setup kiểm tra quyền riêng tư và in step local/remote đã xác minh trước khi tải model.
`SAVE_STEPS` điều khiển lưu local; `CHECKPOINT_SYNC_EVERY_N_SAVES` điều khiển upload đồng bộ.
Checkpoint cuối được upload khi runner thành công; kiểm tra `checkpoint_sync.jsonl` và cảnh báo backup thất bại.
`none` bắt đầu mới; `local`, `remote`, `auto`, hoặc `RESUME_FROM_CHECKPOINT` là đường dẫn chính xác.
Giữ nguyên dữ liệu/cấu hình khi resume vào output mới. Smoke không kết nối Hub.

Có thể dùng **Save Version → Save & Run All** và đóng trình duyệt. `/kaggle/working` là tạm thời;
chỉ checkpoint đã upload hoàn tất mới có thể phục hồi. Các optimizer step chưa lưu/upload có thể mất.
Không xóa checkpoint tự động. Notebook 05 chỉ đọc artifacts; không đồng bộ ngược checkpoint.
Chưa xác minh Kaggle GPU, NF4 hoặc upload Hub thực tế trong lần triển khai này.
